# Pick’em Auditor — Daily Cloud Inference

**Production:** Engine A (history-v5), Engine B (scouting-v7), guarded blend, empirical-Bayes A/B shadow models, Monte Carlo, and market comparisons. Also runs **cloud-only incremental Bayesian player-rate and prospective calibration posteriors**. Always targets the current Toronto date.

**Resources:** Deepnote Basic CPU (2 vCPU / 5 GiB). No Dell runner, no GitHub Actions training, no bets, no automatic model promotions, and no writes to GitHub.

**One-time secure setup:** In Deepnote open **Integrations** → **Environment variables**. Add GITHUB_TOKEN (fine-grained read-only Contents permission on the private royalexx/pickem-auditor repo) and PROPLINE_API_KEY. Attach the integration to this project. Do **not** paste tokens into notebook code or chat. The native Deepnote GitHub import integration may be plan-restricted; this downloads read-only GitHub API archives from the private repository using encrypted credentials.

**Outputs:** Per-run audit JSON/CSV under pickem_runs/YYYY-MM-DD/TIMESTAMP. Lists top 10 per sport showing Engine A, Engine B, Bayesian A, Bayesian B, Online Bayesian A/B, Online Player Rate, Market, and conservative model-minus-market gap. Existing four-model scores remain the screening reference; the online forecasts have no Tier authority. All ranks remain research-only without verified source evidence and calibration.

**Important:** Empirical-Bayes shadow models are not the new hierarchical NUTS models. Automatic NUTS adoption is intentionally disabled until compatible posterior artifacts and convergence/holdout receipts can be validated.

**Daily update:** After the morning settles completed games and commits source-verified evidence to the private GitHub live-data branch, it incrementally updates a separate Gamma-Poisson player-count posterior and a Bayesian logit-calibration posterior. Each afternoon loads only the latest prior-day posterior; same-day outcomes can never leak into the slate. A one-time historical bootstrap reads the private, SHA-pinned population warehouse without recollecting it. Mondays perform a full replay of immutable online evidence for drift detection. This is not a refit or mutation of the full NUTS player/team/season/opponent model.

**Deployment policy:** All online forecasts are research-only. 30-day prospective Brier/log-loss and date-cluster bootstrap diagnostics must support an independent validation review before any production model is changed. Heavy joint NUTS refits require a separately frozen, explicitly approved training campaign; nothing is launched on the Dell or a self-hosted runner.

**Converged precision research shadows:** The pinned NHL saves posterior and NFL targets v3 inference release are evaluated after the afternoon snapshot is frozen, without retraining or promotion. The next morning, the original archived posteriors are compared against verified completed games. Research target-count projections are separate from authentic posted targets lines; unavailable markets are not fabricated.\n

In [ ]:
# Set auto, afternoon or morning. The live Deepnote notebook uses an input selector.
run_phase = 'auto'


In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import os, shutil, sys
NOW_ET = datetime.now(ZoneInfo("America/Toronto"))
run_phase = str(globals().get("run_phase") or "auto").strip().lower()
if run_phase == "auto":
    RUN_PHASE = "morning" if NOW_ET.hour < 12 else "afternoon"
elif run_phase in ("morning","afternoon"):
    RUN_PHASE = run_phase
else:
    raise ValueError("Run phase must be auto, morning, or afternoon")
RUN_DATE = (NOW_ET.date()-timedelta(days=1)).isoformat() if RUN_PHASE=="morning" else NOW_ET.date().isoformat()
print("PICKEM_DEEPNOTE", {
    "phase":RUN_PHASE,"target_game_date":RUN_DATE,"toronto_now":NOW_ET.isoformat(),
    "python":sys.version.split()[0],"git_present":bool(shutil.which("git")),
    "github_secret_configured":bool(os.environ.get("GITHUB_TOKEN")),
    "propline_secret_configured":bool(os.environ.get("PROPLINE_API_KEY")),
})
required=["GITHUB_TOKEN"]+(["PROPLINE_API_KEY"] if RUN_PHASE=="afternoon" else [])
missing=[name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError("Configure encrypted Deepnote Environment variables: "+", ".join(missing))
print("Private GitHub Contents read/write permission required to publish immutable daily files.")


In [ ]:
import os, pathlib, subprocess, sys, tempfile, json, time, shutil, zipfile, urllib.request
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
TARGET_DATE = RUN_DATE
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = pathlib.Path.cwd()/"pickem_runs"/TARGET_DATE/RUN_STAMP
(OUT/"logs").mkdir(parents=True, exist_ok=True)
ROOT = pathlib.Path(tempfile.mkdtemp(prefix="pickem_deepnote_checkout_"))/"pickem-auditor"
receipts={}
ENV=os.environ.copy()
missing=[k for k in (("GITHUB_TOKEN",) if RUN_PHASE=="morning" else ("GITHUB_TOKEN","PROPLINE_API_KEY")) if not ENV.get(k)]
if missing: raise RuntimeError("Missing encrypted Deepnote Environment variables: "+", ".join(missing))

def archive_branch(branch, dst):
    url="https://api.github.com/repos/royalexx/pickem-auditor/zipball/"+branch
    req=urllib.request.Request(url,headers={"Authorization":"Bearer "+ENV["GITHUB_TOKEN"],"Accept":"application/vnd.github+json","User-Agent":"pickem-auditor-inference"})
    zip_path=OUT/(branch.replace("/","-")+".zip")
    with urllib.request.urlopen(req,timeout=120) as response, zip_path.open("wb") as output:
        shutil.copyfileobj(response,output)
    with zipfile.ZipFile(zip_path) as archive:
        prefix=archive.namelist()[0].split("/")[0]
        dst.mkdir(parents=True,exist_ok=True)
        for member in archive.infolist():
            parts=pathlib.PurePosixPath(member.filename).parts
            if len(parts)<2 or parts[0]!=prefix or any(p in ("..","") for p in parts): continue
            target=(dst/pathlib.Path(*parts[1:])).resolve()
            if not target.is_relative_to(dst.resolve()): raise ValueError("Unsafe ZIP entry")
            if member.is_dir():target.mkdir(parents=True,exist_ok=True)
            else:
                target.parent.mkdir(parents=True,exist_ok=True)
                with archive.open(member) as src,target.open("wb") as dest:shutil.copyfileobj(src,dest)
    zip_path.unlink(missing_ok=True)

print("Fetching current private repository code and calibration history via authenticated GitHub API")
archive_branch("main",ROOT)
HISTORY=pathlib.Path(tempfile.mkdtemp(prefix="pickem_shadow_store_"))
archive_branch("data/shadow-calibration",HISTORY)
src=HISTORY/"data/prop_snapshots.jsonl"
if not src.is_file() or not src.stat().st_size: raise RuntimeError("Missing authoritative history")
dst=ROOT/"data/prop_snapshots.jsonl"
dst.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(src,dst)
for name in ("bayesian_shadow","bayesian_b_shadow","posterior_mc","verification","morning_runs","market_odds_history","nfl_context"):
    a=HISTORY/"data"/name
    if a.is_dir():shutil.copytree(a,ROOT/"data"/name,dirs_exist_ok=True)

def stage(name,command,required=False,timeout=1200):
    start=time.monotonic();print("START",name,flush=True)
    try:
        run=subprocess.run(command,cwd=ROOT,env=ENV,capture_output=True,text=True,timeout=timeout)
        code,output=run.returncode,(run.stdout or "")+"\n"+(run.stderr or "")
    except subprocess.TimeoutExpired as exc:code,output=124,str(exc)
    except Exception as exc:code,output=125,type(exc).__name__+": "+str(exc)
    for key in ("GITHUB_TOKEN","PROPLINE_API_KEY"):
        if ENV.get(key):output=output.replace(ENV[key],"[REDACTED]")
    (OUT/"logs"/(name+".log")).write_text(output[-160000:])
    receipts[name]={"ok":code==0,"code":code,"seconds":round(time.monotonic()-start,1)}
    print(name,"OK" if code==0 else "FAILED", "elapsed_s",receipts[name]["seconds"])
    if required and code:raise RuntimeError(name+" failed: "+output[-1200:])
    return code==0

stage("install_dependencies",[sys.executable,"-m","pip","install","-q","-r","requirements.txt","zstandard"],required=True,timeout=1200)
import tarfile,zstandard
cache=HISTORY/"data/source_cache.tar.zst"
if not cache.is_file() or not cache.stat().st_size:raise RuntimeError("Historical usage/season cache missing from branch")
with cache.open("rb") as f:
    with zstandard.ZstdDecompressor().stream_reader(f) as reader:
        with tarfile.open(fileobj=reader,mode="r|") as tar:
            for entry in tar:
                parts=pathlib.PurePosixPath(entry.name).parts
                if len(parts)<3 or parts[0]!="data" or parts[1] not in ("usage","season") or not entry.isfile():continue
                target=(ROOT/pathlib.Path(*parts)).resolve()
                if not target.is_relative_to(ROOT.resolve()):raise ValueError("Unsafe cached file")
                target.parent.mkdir(parents=True,exist_ok=True)
                with tar.extractfile(entry) as src,target.open("wb") as dst:shutil.copyfileobj(src,dst)
print("PRIVATE_REPO_AND_HISTORY_RESTORED",RUN_PHASE,TARGET_DATE)

In [ ]:
# The afternoon run freezes forecasts before games. The morning run settles
# ONLY previously frozen forecasts and collects completed-game training rows.
assert (ROOT/"scripts/live_pool_exchange.py").is_file(), "Update the project checkout to the merged shared-data code."

if RUN_PHASE=="afternoon":
    ENV.update(PICKEM_ODDS_QUOTA_GUARD="1",PICKEM_DISABLE_ODDS="0",
               PICKEM_ENABLE_ODDS_API_NHL_BLOCKS="0")
    stage("nhl_boxscores",[sys.executable,"scripts/nhl_collect.py","--days","3"],timeout=600)
    stage("fetch_slate",[sys.executable,"scripts/fetch_slate.py",
                        "--out","data/slate_fetch.jsonl"],required=True,timeout=1800)
    stage("verify_sports",[sys.executable,"scripts/verify_sports.py",
                          "--slot","afternoon"],timeout=300)
    stage("market_coverage",[sys.executable,"scripts/market_coverage_audit.py",
                             "--quotes","data/odds_quotes.jsonl",
                             "--slate","data/slate_fetch.jsonl",
                             "--out","reports/market_coverage.json"],timeout=300)
    stage("primary_source_research",[sys.executable,"scripts/auto_research.py"],timeout=800)
    stage("engine_A_B_and_guarded_blend",[sys.executable,"scripts/jobs.py",
          "snapshot","--date",TARGET_DATE,"--reuse-collected"],required=True,timeout=1800)
else:
    # This pulls only yesterday's untouched pregame forecasts from our private
    # append-only live-data branch, NEVER recomputes them using today's results.
    # Low-cost smoke test before touching original frozen forecast rows.
    stage("nhl_settlement_unit_tests",[sys.executable,"-m","unittest","discover",
          "-s","tests","-p","test_*settle*.py","-v"],required=True,timeout=120)
    PRED="data/morning_prior_predictions.jsonl"
    stage("restore_yesterday_forecasts",[sys.executable,"scripts/live_pool_exchange.py",
        "restore-forecasts","--before",NOW_ET.date().isoformat(),"--after",TARGET_DATE,
        "--dest",PRED],required=True,timeout=750)
    # Count identifiers without printing player names or private sources.
    _frozen_rows=[__import__("json").loads(line) for line in
                  (ROOT/PRED).read_text().splitlines() if line.strip()]
    _nhl_rows=[r for r in _frozen_rows if r.get("sport")=="NHL"]
    print("NHL_FROZEN_IDENTITY_AUDIT",{
       "rows":len(_nhl_rows),
       "player_id":sum(bool(r.get("player_id")) for r in _nhl_rows),
       "both_teams":sum(bool(r.get("team")) and bool(r.get("opponent")) for r in _nhl_rows),
       "nhl_game_ids":sum(str(r.get("event_id_namespace") or "").lower()=="nhl"
                           for r in _nhl_rows)})
    # Collect completed gamecenter/official player-game data BEFORE settling.
    # NHL grading uses exactly these source-verified population records.
    stage("nhl_completed_games",[sys.executable,"scripts/nhl_collect.py",
          "--date",NOW_ET.date().isoformat(),"--days","3"],timeout=750)
    stage("collect_population",[sys.executable,"scripts/live_pool_exchange.py",
          "collect","--day",TARGET_DATE,"--sports","NHL,NBA,NFL"],
          timeout=2400)
    from pathlib import Path
    pregame=ROOT/PRED
    if pregame.is_file() and pregame.stat().st_size:
        stage("settle_yesterday",[sys.executable,"scripts/jobs.py","settle",
              "--as-of",NOW_ET.date().isoformat(),"--snapshots",PRED],
              required=True,timeout=1800)
        # A successful process exit is not evidence that NHL markets settled.
        import json as _settlement_json
        _source_rows=[_settlement_json.loads(line) for line in pregame.read_text().splitlines()
                      if line.strip()]
        _nhl_original=sum(r.get("sport")=="NHL" for r in _source_rows)
        _nhl_settled=sum(r.get("sport")=="NHL" and r.get("actual_stat") is not None
                         and r.get("outcome") in ("win","loss","push") for r in _source_rows)
        print("NHL_MORNING_SETTLEMENT",{"original":_nhl_original,"settled":_nhl_settled,
                                         "unresolved":_nhl_original-_nhl_settled})
        if _nhl_original and not _nhl_settled:
            raise RuntimeError("NHL_SETTLEMENT_ZERO_MATCHES_REQUIRES_SOURCE_AUDIT")
    else:
        print("NO_PRIOR_FORECASTS_TO_SETTLE: still collect official game results")
print("PICKEM_PHASE_STAGE_STATUS",RUN_PHASE,receipts)

In [ ]:
if RUN_PHASE=="afternoon":
    SNP = "data/prop_snapshots.jsonl"
    if not (ROOT/SNP).is_file():
        raise RuntimeError("Forecast history missing; no Bayesian comparisons possible.")
    stage("bayesian_A_shadow",[sys.executable,"scripts/bayesian_shadow.py","--date",TARGET_DATE,
        "--snapshots",SNP,"--data-root","data","--out",f"data/bayesian_shadow/{TARGET_DATE}.jsonl",
        "--receipt","reports/bayesian_a_shadow_receipt.json"],timeout=800)
    stage("bayesian_B_shadow",[sys.executable,"scripts/bayesian_b_shadow.py","--date",TARGET_DATE,
        "--snapshots",SNP,"--data-root","data","--out",f"data/bayesian_b_shadow/{TARGET_DATE}.jsonl",
        "--receipt","reports/bayesian_b_shadow_receipt.json"],timeout=800)
    stage("posterior_Monte_Carlo",[sys.executable,"scripts/posterior_mc_shadow.py","--date",TARGET_DATE,
        "--original",SNP,"--data-root","data","--out",f"data/posterior_mc/{TARGET_DATE}.jsonl",
        "--receipt","reports/posterior_mc_receipt.json"],timeout=800)
    # Independently trained, fully converged posteriors. Research sidecars
    # only; they never alter the four-model ranking or any source snapshot.
    # Downloads use pinned GitHub releases, not new training or paid compute.
    # Isolate numerical packages so Deepnote's live pandas/NumPy kernel
    # cannot be changed by installing ArviZ/NetCDF inference dependencies.
    nuts_env=OUT/"precision_inference_venv"
    nuts_py=str(nuts_env/"bin/python")
    nuts_ok=stage("nuts_inference_venv",[sys.executable,"-m","venv",
                   "--system-site-packages",str(nuts_env)],timeout=180)
    if nuts_ok:
        nuts_ok=stage("nuts_inference_dependencies",[nuts_py,"-m","pip","install","-q",
           "numpy>=1.26,<2","scipy>=1.14,<2","arviz>=0.18,<1",
           "h5netcdf>=1.2,<2","h5py>=3.9,<4",
           "-r","requirements-nfl-inference.txt"],timeout=900)
    if nuts_ok:
        saved=stage("nhl_saves_release",[sys.executable,"scripts/precision_shadow_release.py",
                     "--model","saves","--out","data/private_models/nhl_saves",
                     "--receipt","reports/nhl_saves_release_download.json"],timeout=700)
        if saved:
            stage("nhl_saves_nuts_shadow",[nuts_py,"scripts/nhl_saves_posterior_shadow.py",
                  "--snapshots",SNP,"--model-dir","data/private_models/nhl_saves",
                  "--date",TARGET_DATE,"--out",f"data/nhl_joint_saves_shadow/{TARGET_DATE}.jsonl",
                  "--receipt","reports/nhl_joint_saves_shadow_receipt.json"],timeout=800)
        targets=stage("nfl_targets_release",[sys.executable,"scripts/precision_shadow_release.py",
                     "--model","targets","--out","data/private_models/nfl_targets",
                     "--receipt","reports/nfl_targets_release_download.json"],timeout=700)
        if targets:
            stage("nfl_targets_nuts_shadow",[nuts_py,"scripts/nfl_context_daily.py","run",
                  "--date",TARGET_DATE,"--snapshots",SNP,"--data-root","data",
                  "--model-dir","data/private_models/nfl_targets",
                  "--out",f"data/nfl_context/{TARGET_DATE}.jsonl",
                  "--receipt","reports/nfl_context_daily.json"],timeout=900)
    else:
        print("NUTS_SHADOWS_UNAVAILABLE: inference dependencies not installed; production unchanged")

    stage("research_triage",[sys.executable,"scripts/research_triage.py","--date",TARGET_DATE,
        "--path",SNP,"--packet",f"data/research_{TARGET_DATE}.json",
        "--out","reports/research_triage.json"],timeout=360)
    if (ROOT/"reports/research_triage.json").exists():
        stage("research_ranked_board",[sys.executable,"scripts/research_ranked_board.py",
            "--date",TARGET_DATE,"--snapshots",SNP,"--queue","reports/research_triage.json",
            "--archive-root","data/verification","--out","reports/research_ranked_board.json"],timeout=360)
    stage("verified_board",[sys.executable,"scripts/verified_board.py","--date",TARGET_DATE,
        "--snapshots",SNP,"--archive-root","data/verification",
        "--out","reports/verified_board.json"],timeout=360)
else:
    print("MORNING_MODE: Bayesian pregame sidecars cannot be regenerated after the game.")
    SNP = "data/morning_prior_predictions.jsonl"


In [ ]:
# New cloud-only fifth Bayesian model. It is a SHADOW challenger and never
# changes our original four model probabilities, production tiers, or NUTS.
# Current-date forecasts can only read states trained through an earlier day.
if RUN_PHASE=="afternoon":
    stage("online_bayesian_shadow",[sys.executable,
        "scripts/cloud_bayes_incremental.py","shadow",
        "--date",TARGET_DATE,"--root",str(ROOT),
        "--out",f"data/online_bayes_shadow/{TARGET_DATE}.jsonl",
        "--report","reports/cloud_bayes_shadow_receipt.json"],
        required=True,timeout=900)
else:
    print("Morning mode: online posterior update occurs only AFTER official settlement and Git publication.")


In [ ]:
if RUN_PHASE=="afternoon":
    # Show the real four-model probabilities for the latest same-day forecast batch.
    import pandas as pd
    from IPython.display import display

    def load_jsonl(path):
        if not path.is_file(): return []
        result=[]
        with path.open(encoding="utf-8",errors="replace") as handle:
            for line in handle:
                try:
                    if line.strip():result.append(json.loads(line))
                except (ValueError,TypeError):pass
        return result

    def probability(value):
        return float(value) if type(value) in (int,float) and 0 <= value <= 1 else None

    all_today=[r for r in load_jsonl(ROOT/SNP)
               if r.get("game_date")==TARGET_DATE and r.get("run_type")=="shadow"]
    batch_key=lambda r:(str(r.get("forecast_at") or ""),str(r.get("run_id") or ""))
    latest_run=max(all_today,key=batch_key).get("run_id") if all_today else None
    rows=[r for r in all_today if r.get("run_id")==latest_run]

    def prop_key(r):
        return (str(r.get("run_id") or ""),str(r.get("sport") or ""),
                str(r.get("player_id") or r.get("player") or ""),
                str(r.get("stat") or ""),str(r.get("side") or ""),
                str(r.get("reference_line") if r.get("reference_line") is not None else ""))

    indices={}
    for engine, subdir in (("bayes_A","bayesian_shadow"),("bayes_B","bayesian_b_shadow")):
        src=load_jsonl(ROOT/"data"/subdir/(TARGET_DATE+".jsonl"))
        indices[engine]={prop_key(r):r for r in src if r.get("run_id")==latest_run}
    mc_index={prop_key(r):r for r in load_jsonl(ROOT/"data/posterior_mc"/(TARGET_DATE+".jsonl"))
              if r.get("run_id")==latest_run}

    records=[]
    seen=set()
    for r in rows:
        key=prop_key(r)
        if key in seen:continue
        seen.add(key)
        A=r.get("engine_a") or {}
        B=r.get("engine_b") or {}
        blend=r.get("blend") or {}
        market=r.get("market") or {}
        a=probability(A.get("q_win_robust"))
        b=probability(B.get("q_win"))
        bayes_A=indices["bayes_A"].get(key,{})
        bayes_B=indices["bayes_B"].get(key,{})
        ba=probability(bayes_A.get("q_win")) if bayes_A.get("status")=="forecast" else None
        bb=probability(bayes_B.get("q_win")) if bayes_B.get("status")=="forecast" else None
        m=probability(r.get("q_market_comparable"))
        if m is None:m=probability(market.get("q_market_comparable"))
        estimates=(a,b,ba,bb)
        minimum=min(estimates) if all(p is not None for p in estimates) else None
        reliable=(A.get("reliable") is True and B.get("reliable") is True
                  and not B.get("abstain"))
        valid_market=(m is not None and market.get("market_valid") is not False)
        screen=(minimum is not None and reliable and valid_market and minimum>=m+0.03)
        mc=mc_index.get(key,{})
        records.append({
            "sport":r.get("sport"),"player":r.get("player"),"team":r.get("team"),
            "opponent":r.get("opponent"),"stat":r.get("stat"),
            "side":r.get("side"),"line":r.get("reference_line"),
            "Engine_A":a,"Engine_B":b,"Bayesian_A":ba,"Bayesian_B":bb,
            "Market":m,"Conservative_gap_pp":100*(minimum-m) if minimum is not None and m is not None else None,
            "lowest_model":minimum,
            "70_30_diagnostic":0.7*a+0.3*b if a is not None and b is not None and reliable else None,
            "validated_blend":probability(blend.get("q_win")) if blend.get("validated") is True else None,
            "production_engines_reliable":reliable,"market_valid":valid_market,
            "four_model_screen":bool(screen),
            "original_rank":r.get("final_rank"),"block_reason":r.get("reason"),
            "forecast_at":r.get("forecast_at"),
            "Bayesian_A_status":bayes_A.get("status"),
            "Bayesian_B_status":bayes_B.get("status"),
            "MC_A_agreement":(mc.get("engines") or {}).get("A",{}).get("analytic_agreement"),
            "MC_B_agreement":(mc.get("engines") or {}).get("B",{}).get("analytic_agreement"),
        })
    frame=pd.DataFrame.from_records(records)
    if frame.empty:
        print("NO_SAME_DAY_FORECASTS: inspect the stage logs.")
    else:
        frame.to_csv(OUT/"all_four_model_comparisons.csv",index=False)
        columns=["player","stat","side","line","Engine_A","Engine_B",
                 "Bayesian_A","Bayesian_B","Market","Conservative_gap_pp","block_reason"]
        for sport in ("NFL","NBA","NHL"):
            screened=frame[(frame.sport==sport)&(frame.four_model_screen==True)].copy()
            screened=screened.sort_values(["Conservative_gap_pp","lowest_model"],
                                          ascending=False).head(10)
            screened.to_csv(OUT/(sport+"_top10_four_model_research.csv"),index=False)
            print(sport, "screened research candidates",len(screened),"— NOT verified picks")
            if not screened.empty:
                display(screened[columns].round(4))
        print("Coverage by sport and market")
        display(frame.groupby(["sport","stat"],dropna=False).agg(
            forecasts=("player","size"),four_model_screen=("four_model_screen","sum")).reset_index())
else:
    import pandas as pd
    rows = []
    frame = pd.DataFrame()
    latest_run = None
    print("MORNING_MODE: all historical predictions remain frozen; scoring is handled separately below.")


In [ ]:
# Include online-updated Bayesian forecasts alongside the original four models.
# The conservative gap and ranking screen still use the four established
# research outputs; the online posterior has no promotion authority.
if RUN_PHASE=="afternoon" and not frame.empty:
    import pandas as pd
    raw=load_jsonl(ROOT/"data/online_bayes_shadow"/(TARGET_DATE+".jsonl"))
    online_lookup={}
    for item in raw:
        key=(str(item.get("sport")),str(item.get("player")),
             str(item.get("stat")),str(item.get("side")),
             str(item.get("reference_line")))
        if key in online_lookup and online_lookup[key]!=item:
            raise ValueError("AMBIGUOUS_ONLINE_BAYESIAN_PLAYER_MARKET_IDENTITY")
        online_lookup[key]=item

    def updated_prob(row,model):
        key=(str(row.sport),str(row.player),str(row.stat),str(row.side),
             str(row.line))
        item=online_lookup.get(key,{})
        values=(item.get("models") or {}).get(model,{})
        if values.get("status") in ("research_shadow","warmup"):
            return values.get("q_win")
        return None

    for model,col in (
            ("bayesian_a_online_calibrated","Online_Bayesian_A"),
            ("bayesian_b_online_calibrated","Online_Bayesian_B"),
            ("player_rate_gamma_poisson","Online_Player_Rate")):
        frame[col]=frame.apply(lambda row:updated_prob(row,model),axis=1)

    frame.to_csv(OUT/"four_plus_online_models.csv",index=False)
    print("ONLINE_BAYESIAN_SHADOW: exploratory fifth-family estimates; not eligible for Tier 1/2")
    fields=["player","stat","side","line","Engine_A","Engine_B",
            "Bayesian_A","Bayesian_B","Online_Bayesian_A","Online_Bayesian_B",
            "Online_Player_Rate","Market","Conservative_gap_pp"]
    for sport in ("NFL","NBA","NHL"):
        ranked=frame[(frame.sport==sport)&(frame.four_model_screen==True)].copy()
        ranked=ranked.sort_values("Conservative_gap_pp",ascending=False).head(10)
        ranked.to_csv(OUT/(sport+"_four_plus_online_top10.csv"),index=False)
        print(sport,"four-model screen + online challenger",len(ranked))
        if not ranked.empty:
            display(ranked[fields].round(4))


In [ ]:
# Morning: audit ONLY immutable pregame probabilities against official settled results.
# A prediction made after kickoff is not eligible for prospective validation.
morning_report=None
if RUN_PHASE=="morning":
    import math, sys, json
    sys.path.insert(0,str(ROOT/"scripts"))
    import live_pool_exchange as pool
    from collections import defaultdict
    before=NOW_ET.date().isoformat()
    originals=[]
    for doc,path in pool.iter_remote_docs(token=ENV["GITHUB_TOKEN"],
                                          before=before,after=TARGET_DATE,phase="afternoon"):
        originals.append(doc)

    def key(r):
        return (str(r.get("run_id") or ""),str(r.get("sport") or ""),
                str(r.get("player_id") or r.get("player") or ""),
                str(r.get("stat") or ""),str(r.get("side") or ""),
                str(r.get("reference_line") if r.get("reference_line") is not None else ""))

    # Use the identical earliest archived cohort restored for settlement.
    # Later reruns stay immutable in GitHub but cannot change today's audit.
    canonical_originals=pool.canonical_afternoon_docs(originals)
    later_archives_excluded=len(originals)-len(canonical_originals)
    originals=[canonical_originals[d] for d in sorted(canonical_originals)]

    forecast_index={}
    sidecars={"bayesian_a":{},"bayesian_b":{},"online_bayes":{}}
    for doc in originals:
        for r in doc.get("forecasts",[]):
            k=key(r)
            if k in forecast_index and forecast_index[k]!=r:
                raise ValueError("conflicting_immutable_pregame_forecast")
            forecast_index[k]=r
        for name in sidecars:
            for r in (doc.get("shadow") or {}).get(name,[]):
                k=key(r)
                if k in sidecars[name] and sidecars[name][k]!=r:
                    raise ValueError("conflicting_immutable_bayesian_forecast")
                sidecars[name][k]=r

    settled=pool.jsonl(ROOT/"data/morning_prior_predictions.jsonl")
    scored=defaultdict(list)
    excluded=defaultdict(int)
    stats=[]
    for row in settled:
        k=key(row)
        orig=forecast_index.get(k)
        if orig is None or orig.get("actual_stat") is not None:
            excluded["original_forecast_unavailable"]+=1
            continue
        try:
            forecast=datetime.fromisoformat(str(orig.get("forecast_at")).replace("Z","+00:00"))
            start=datetime.fromisoformat(str(orig.get("event_start")).replace("Z","+00:00"))
            if forecast.tzinfo is None or start.tzinfo is None or forecast>=start:
                excluded["not_verified_pregame"]+=1
                continue
        except (TypeError, ValueError):
            excluded["start_time_or_forecast_timestamp_missing"]+=1
            continue
        status=row.get("outcome")
        if status=="push":
            excluded["push_excluded_from_win_loss_scoring"]+=1
            continue
        if status not in ("win","loss") or row.get("actual_stat") is None:
            excluded["unsettled_or_missing_stat"]+=1
            continue
        y=int(status=="win")
        a=orig.get("engine_a") or {}
        b=orig.get("engine_b") or {}
        market=orig.get("market") or {}
        model={
            "engine_a":a.get("q_win_robust"),
            "engine_b":b.get("q_win"),
            "bayesian_a":sidecars["bayesian_a"].get(k,{}).get("q_win"),
            "bayesian_b":sidecars["bayesian_b"].get(k,{}).get("q_win"),
            "market":orig.get("q_market_comparable") or market.get("q_market_comparable"),
        }
        # Each additional online model was generated and frozen in the
        # afternoon. Never recompute past predictions after outcomes arrive.
        online=(sidecars["online_bayes"].get(k,{}) or {}).get("models") or {}
        for label,name in (("online_bayesian_a","bayesian_a_online_calibrated"),
                           ("online_bayesian_b","bayesian_b_online_calibrated"),
                           ("online_player_rate","player_rate_gamma_poisson")):
            item=online.get(name) or {}
            if item.get("status") not in ("research_shadow","warmup"):
                continue
            estimated=item.get("q_win")
            push=item.get("q_push",0)
            try:
                estimated=float(estimated);push=float(push)
                if not (0<=estimated<=1 and 0<=push<1 and estimated+push<=1+1e-9):
                    continue
                model[label]=estimated/(1-push)
            except (ValueError,TypeError,ZeroDivisionError):
                continue
        if a.get("reliable") is True and b.get("reliable") is True:
            try:
                model["diagnostic_70_30"]=0.7*float(model["engine_a"])+0.3*float(model["engine_b"])
            except (TypeError,ValueError):pass
        entry={"game_date":TARGET_DATE,"sport":row.get("sport"),"stat":row.get("stat"),
               "player":row.get("player"),"side":row.get("side"),
               "line":row.get("reference_line"),"actual_stat":row.get("actual_stat"),
               "outcome":status,"forecast_at":orig["forecast_at"]}
        for name,p in model.items():
            try:
                p=float(p)
                if name=="bayesian_a":
                    src=sidecars["bayesian_a"].get(k) or {}
                    push=float(src.get("q_push") or 0)
                    if 0<=push<1:p=p/(1-push)
                elif name=="bayesian_b":
                    src=sidecars["bayesian_b"].get(k) or {}
                    push=float(src.get("q_push") or 0)
                    if 0<=push<1:p=p/(1-push)
                elif name=="engine_a":
                    push=float(a.get("q_push_robust") or 0)
                    if 0<=push<1:p=p/(1-push)
                elif name=="engine_b":
                    push=float(b.get("q_push") or 0)
                    if 0<=push<1:p=p/(1-push)
                if not math.isfinite(p) or not 0<=p<=1:continue
            except (ValueError,TypeError):continue
            brier=(p-y)**2
            loss=-math.log(max(1e-12,p if y else 1-p))
            scored[(row.get("sport"),row.get("stat"),name)].append((p,y,brier,loss))
            entry[name]=p
        stats.append(entry)
    summary={}
    for (sport,stat,name),v in sorted(scored.items()):
        summary.setdefault(sport,{}).setdefault(stat,{})[name]={
            "n":len(v),"avg_probability":round(sum(x[0] for x in v)/len(v),5),
            "observed_hit_rate":round(sum(x[1] for x in v)/len(v),5),
            "brier":round(sum(x[2] for x in v)/len(v),6),
            "log_loss":round(sum(x[3] for x in v)/len(v),6)}
    morning_report={"schema":"deepnote_daily_prospective_model_audit_v1",
                    "date":TARGET_DATE,"source_forecast_bundles":len(originals),
                    "original_forecasts":len(forecast_index),
                    "settled_rows":sum(x.get("outcome") in ("win","loss","push") for x in settled),
                    "scored_nonpush_rows":len(stats),"exclusions":dict(excluded),
                    "metrics":summary,
                    "validated_for_deployment":False,
                    "warning":"Descriptive only until chronological holdout meets sample thresholds"}
    (OUT/"morning_model_validation.json").write_text(json.dumps(morning_report,indent=2))
    pool.dump_jsonl(OUT/"settled_model_predictions.jsonl",stats)
    # The independently trained SOG/saves/targets posteriors remain outside
    # historical Bayesian A/B scoring; replay only archived pre-event sidecars.
    stage("precision_nuts_morning_validation",[sys.executable,
          "scripts/precision_shadow_scoring.py","--day",TARGET_DATE,
          "--root",str(ROOT),
          "--out",str(OUT/"precision_nuts_model_validation.json"),
          "--predictions-out",str(OUT/"precision_nuts_settled_predictions.jsonl")],
          timeout=700)
    print("PROSPECTIVE_AUDIT",json.dumps({k:v for k,v in morning_report.items() if k!="metrics"},indent=2))
    for sport,markets in sorted(summary.items()):
        print(sport,"sample sizes",{
            stat:metrics.get("engine_a",{}).get("n",0) for stat,metrics in markets.items()})
else:
    print("AFTERNOON: no postgame statistics consumed; morning settlement performed separately.")


In [ ]:
# Discovery only: do NOT use unfinished or unvalidated hierarchical NUTS posterior files.
from urllib.request import Request, urlopen

nuts_discovery={"status":"not_checked","new_model_inference_enabled":False}
try:
    token=os.environ.get("GITHUB_TOKEN")
    headers={"Authorization":"Bearer "+token,"Accept":"application/vnd.github+json",
             "User-Agent":"pickem-auditor-inference"}
    url="https://api.github.com/repos/royalexx/pickem-auditor/actions/runs?per_page=30"
    with urlopen(Request(url,headers=headers),timeout=25) as response:
        runs=json.load(response).get("workflow_runs",[])
    training=[r for r in runs if "Precision 31-market research campaign" in (r.get("name") or "")]
    if training:
        recent=training[0]
        nuts_discovery={"status":recent.get("status"),"conclusion":recent.get("conclusion"),
                        "workflow_run_id":recent.get("id"),"new_model_inference_enabled":False}
    else:
        nuts_discovery={"status":"no_training_campaign_found","new_model_inference_enabled":False}
except Exception as e:
    nuts_discovery={"status":"status_query_unavailable","error_type":type(e).__name__,
                    "new_model_inference_enabled":False}
print("NUTS_DISCOVERY_RESEARCH_ONLY",nuts_discovery)

# Copy the compact source receipts, not the private historical archive or credentials.
summary={"date":TARGET_DATE,"run_stamp":RUN_STAMP,
         "environment":"Deepnote_Basic_CPU","source_repo":"royalexx/pickem-auditor",
         "latest_run_id":latest_run,"forecast_rows":len(rows),
         "stages":receipts,
         "research_candidates":({k:int(v) for k,v in frame.loc[
             frame.four_model_screen==True,"sport"].value_counts().items()}
             if not frame.empty else {}),
         "new_hierarchical_nuts":nuts_discovery,
         "publication_eligible":False}
summary["converged_posteriors"]={
    "nhl_saves":receipts.get("nhl_saves_nuts_shadow",{}),
    "nfl_targets":receipts.get("nfl_targets_nuts_shadow",{}),
    "shadow_only":True,
    "production_weights_changed":False}
for p in (ROOT/"reports").glob("*.json"):
    if p.stat().st_size<3_000_000:
        shutil.copy2(p,OUT/p.name)
for subdir in ("bayesian_shadow","bayesian_b_shadow","posterior_mc","online_bayes_shadow",
               "nhl_joint_saves_shadow","nfl_context"):
    p=ROOT/"data"/subdir/(TARGET_DATE+".jsonl")
    if p.exists() and p.stat().st_size<15_000_000:
        shutil.copy2(p,OUT/(subdir+".jsonl"))
(OUT/"receipt.json").write_text(json.dumps(summary,indent=2,default=str))
print("AUDIT_READY",{
    "date":TARGET_DATE,"latest_run_id":latest_run,
    "forecast_rows":len(rows),"research_candidates":summary["research_candidates"],
    "all_stages_succeeded":all(v.get("ok") for v in receipts.values()),
    "published_approved_picks":False,
    "outputs":str(OUT)})
print("This is a research report. Compare probabilities and verify status/lines before using any selection.")

# Append an immutable, source-linked compressed daily bundle to GitHub.
# Never modify the frozen precision campaign or the existing calibration branch.
from pathlib import Path
if RUN_PHASE=="afternoon":
    command=[sys.executable,"scripts/live_pool_exchange.py","publish",
             "--phase","afternoon","--day",TARGET_DATE,
             "--snapshots","data/prop_snapshots.jsonl"]
else:
    command=[sys.executable,"scripts/live_pool_exchange.py","publish",
             "--phase","morning","--day",TARGET_DATE,
             "--snapshots","data/morning_prior_predictions.jsonl",
             "--population",f"data/live_pool_staging/populations/{TARGET_DATE}.jsonl"]
stage("publish_to_shared_git_data_pool",command,required=True,timeout=1000)
summary["run_phase"]=RUN_PHASE
summary["github_data_branch"]="data/live-observations"
summary["shared_pool_published"]=receipts["publish_to_shared_git_data_pool"]["ok"]
summary["morning_validation"]=(
    {k:v for k,v in morning_report.items() if k!="metrics"}
    if morning_report is not None else None)
(OUT/"receipt.json").write_text(json.dumps(summary,indent=2,default=str))
print("GITHUB_LIVE_POOL_PUBLICATION_SUCCEEDED",RUN_PHASE,TARGET_DATE)


# Chronological Bayesian state update is cloud-only and ONLY on morning
# completed games. Afternoon inference never updates a posterior from a
# target day's scores. Monday runs independently replay the cloud archive.
if RUN_PHASE=="morning":
    update_cmd=[sys.executable,"scripts/cloud_bayes_incremental.py","update",
       "--as-of",NOW_ET.date().isoformat(),
       "--report","reports/cloud_bayes_monitor.json"]
    if NOW_ET.weekday()==0:
        update_cmd.append("--weekly-rebuild")
    stage("bayesian_incremental_morning",update_cmd,required=True,timeout=2400)
    monitor_path=ROOT/"reports/cloud_bayes_monitor.json"
    if monitor_path.exists():
        monitor=json.loads(monitor_path.read_text())
        summary["online_bayesian_update"]={
            "status":monitor.get("status"),
            "through":monitor.get("through"),
            "historical_seed_events":monitor.get("historical_seed_events"),
            "population_events":monitor.get("population_events"),
            "calibration_events":monitor.get("calibration_events"),
            "weekly_full_replay_checked":monitor.get("weekly_full_replay_checked"),
            "new_nuts_fit_started":False,
            "production_weights_changed":False}
        (OUT/"cloud_bayes_monitor.json").write_text(json.dumps(monitor,indent=2))
        scores=(monitor.get("monitor") or {}).get("markets") or {}
        print("PROSPECTIVE_ONLINE_BAYESIAN_MONITOR",json.dumps(scores,indent=2)[:18000])
else:
    summary["online_bayesian_update"]={
        "status":"loaded_prior_posterior_for_shadow_inference",
        "no_future_data_used":True,"new_nuts_fit_started":False,
        "production_weights_changed":False}
summary["stages"]=receipts
(OUT/"receipt.json").write_text(json.dumps(summary,indent=2,default=str))
print("PICKEM_CLOUD_INFERENCE_ARCHIVED",RUN_PHASE,TARGET_DATE,
      "git_data_published",receipts["publish_to_shared_git_data_pool"]["ok"])
